In [9]:
import torch
import joblib
import captum
from goodpoints import compress
from bonXAI.core.data_loader import DataLoader

# --- 1. Your existing setup code (remains the same) ---
loader = DataLoader()
dataset_id = 554
X_train, y_train, X_test, y_test, model, _ = loader.load_from_openml(dataset_id=dataset_id, model_name="ann", task_type=None)
X_test = X_test[:200] 

ids = compress.compresspp_kt(X_test, kernel_type=b"gaussian", g=4)
X_compressed = X_test[ids]

explainer = captum.attr.IntegratedGradients(model.model_)
inputs = torch.as_tensor(X_test, dtype=torch.float32)
baselines = torch.as_tensor(X_compressed, dtype=torch.float32)

# --- 2. Calculate and collect explanations for all classes ---

num_classes = model.model_.network[-1].out_features
print(f"Number of classes: {num_classes}")  


explanations_per_class = [] # Use a list to store results

for class_index in range(num_classes):
    print(f"Calculating explanations for class {class_index}...")
    
    # This part is the same as before
    tasks = [
        joblib.delayed(explainer.attribute)(inputs, baselines[[i]], target=class_index) 
        for i in range(baselines.shape[0])
    ]
    results = joblib.Parallel(n_jobs=8)(tasks)
    explanation_for_one_class = torch.mean(torch.stack(results), dim=0)
    
    # Add the result for the current class to our list
    explanations_per_class.append(explanation_for_one_class)

# --- 3. Stack the results into a single tensor ---

# torch.stack creates a new dimension from a list of tensors.
# We specify dim=2 to add the new dimension at the end.
final_explanation = torch.stack(explanations_per_class, dim=2)

# --- 4. Check the final result ---
print("\nShape of the final explanation tensor:", final_explanation.shape)

Early stopping at epoch 20
Number of classes: 10
Calculating explanations for class 0...
Calculating explanations for class 1...
Calculating explanations for class 2...
Calculating explanations for class 3...
Calculating explanations for class 4...
Calculating explanations for class 5...
Calculating explanations for class 6...
Calculating explanations for class 7...
Calculating explanations for class 8...
Calculating explanations for class 9...

Shape of the final explanation tensor: torch.Size([200, 710, 10])
